# Chapter 10: Mission Impossible

**Colab slideshow notebook**  
Based on Chapter 10 of the book "Computational Thinking for Life Scientists"
by Benny Chor and Amir Rubinstein.

## Part V: Limitations of Computing — "Easier said than done"

Computers forecast weather, play chess, recognise faces, diagnose cancer cells, drive cars…

**Are computers omnipotent?** Can every well-defined yes/no problem be solved by a program,
given a talented enough programmer?

**No.** In 1936 — before digital computers existed — Alan Turing proved that some problems
cannot be solved by **any** computer, ever.

- **Chapter 10:** problems that cannot be solved at all
- **Chapter 11:** problems that can be solved in principle, but every known solution is far too slow

## Chapter goals

By the end of this chapter, we want to understand:

- what the **halting problem** is, and why halting matters in practice,
- the **Collatz process**: a tiny program whose halting nobody can prove,
- **Turing's theorem**: no program can solve the halting problem,
- the idea of the proof: a program that runs on (a modified version of) itself.

## 10.1 The halting problem: two almost identical functions

In [ ]:
# Code snippet: stoppy — always halts

def stoppy(n):
    i = 1
    while i <= n:
        print(i)
        i = i + 1
    return "hello, world!"

print("returned:", stoppy(8))

## 10.1 The halting problem
### What will be the output?

In [ ]:
def stoppy(n):
    i = 1
    while i <= n:
        i = i + 1
    return i

print(stoppy(3), stoppy(0))

## Answer
Why: the loop stops when i becomes n + 1; for n = 0 the loop body never runs.

In [ ]:
def stoppy(n):
    i = 1
    while i <= n:
        i = i + 1
    return i

print(stoppy(3), stoppy(0))

4 1


## loopy: one missing line

```python
def loopy(n):
    i = 1
    while i <= n:
        print(i)          # i = i + 1 was dropped...
    return "hello, world!"
```

⚠️ **Do not run `loopy(8)`.** `i` and `n` never change, so the condition stays true forever:
an **infinite loop** (a non-terminating program). You would have to stop it by hand
(in Colab: the ■ stop button / Runtime → Interrupt execution).

Below is a safe version that gives up after a fixed number of iterations.

In [ ]:
# Code snippet: loopy with a safety limit on the number of iterations

def loopy_limited(n, max_iterations=5):
    i = 1
    iterations = 0
    while i <= n:
        print(i)
        iterations += 1
        if iterations == max_iterations:
            return (f"stopped by the safety limit after "
                    f"{max_iterations} iterations")
    return "hello, world!"

print("returned:", loopy_limited(8))

## Why does halting matter?

A program has been running for **six days**. Should we abort it?

- If it will never halt — abort, and free the expensive computer.
- If it would halt with the answer after **seven** days — aborting wastes six days of work.

**The halting problem:** given a program `f` and an input `x`, does `f` halt when run on `x`?

## Exercise 1

*Fast Fox* suggests: "Given `f` and `x`, run `f` on `x`. If it halts, answer *f halts on x*."
What is wrong with this solution? Where does it fail?

In [ ]:
# Code snippet: Fast Fox's idea with a time budget — and why it fails
import sys

def halts_within(f, x, max_lines):
    ''' run f(x) for at most max_lines executed lines of code.
        True  = f halted within the budget
        None  = budget exhausted: we DON'T KNOW whether f would halt '''
    counter = [0]
    class OutOfBudget(Exception): pass
    def tracer(frame, event, arg):
        if event == "line":
            counter[0] += 1
            if counter[0] > max_lines:
                raise OutOfBudget
        return tracer
    sys.settrace(tracer)
    try:
        f(x)
        return True
    except OutOfBudget:
        return None
    finally:
        sys.settrace(None)

def loopy_quiet(n):           # like loopy, without printing
    i = 1
    while i <= n:
        pass
    return "hello, world!"

def slow_but_halts(n):        # halts — but only after about 3n lines
    i = 0
    while i < n:
        i = i + 1
    return "done"

print("slow_but_halts(1_000):    ",
      halts_within(slow_but_halts, 1_000, 10_000))
print("slow_but_halts(1_000_000):",
      halts_within(slow_but_halts, 1_000_000, 10_000))
print("loopy_quiet(8):           ", halts_within(loopy_quiet, 8, 10_000))

## What the experiment shows

- With a budget, "halted" is always a correct answer.
- But "budget exhausted" (`None`) is the **same** for a program that never halts (`loopy_quiet`)
  and for a program that simply needs more time (`slow_but_halts(1_000_000)`).
- No fixed budget can tell them apart. Running the program is **not** a solution to the halting problem.

## 10.2 The Collatz conjecture

Start from any integer `n ≥ 1`:

- if `n` is **even** → divide it by 2
- if `n` is **odd** → replace it by `3n + 1`
- stop when you reach **1**

In [ ]:
# Code snippet: the Collatz process

def collatz(n):
    ''' runs the Collatz iteration and prints each element till 1 is reached.
        Will not halt if 1 is never reached '''
    initial = n
    print(initial)
    while n != 1:
        if n % 2 == 0:
            n = n // 2
        else:
            n = 3 * n + 1
        print(n)
    # if we got here, n must be 1
    print("halted on initial input", initial)

collatz(64)

## 10.2 The Collatz conjecture

In [ ]:
# Code snippet: Collatz on 9 and on 13 — up and down, hard to predict

collatz(9)
print()
collatz(13)

## 10.2 The Collatz conjecture

In [ ]:
# Code snippet: count the steps instead of printing them

def collatz_steps(n):
    ''' number of Collatz steps from n to 1 (and the highest value reached) '''
    steps, highest = 0, n
    while n != 1:
        n = n // 2 if n % 2 == 0 else 3 * n + 1
        steps += 1
        highest = max(highest, n)
    return steps, highest

for n in [13, 27, 512, 2**50]:
    steps, highest = collatz_steps(n)
    print(f"n = {n:>20,}: {steps:4} steps, highest value {highest:,}")

## 10.2 The Collatz conjecture

In [ ]:
# Code snippet: number of steps for n = 1 ... 10,000 — no simple pattern
import matplotlib.pyplot as plt

ns = range(1, 10_001)
plt.figure(figsize=(9, 3.5))
plt.scatter(ns, [collatz_steps(n)[0] for n in ns], s=1)
plt.xlabel("n"); plt.ylabel("steps to reach 1")
plt.title("Collatz process: number of steps")
plt.show()

## 10.2 The Collatz conjecture
### What will be the output?

In [ ]:
n = 6
seq = [n]
while n != 1:
    n = n // 2 if n % 2 == 0 else 3 * n + 1
    seq.append(n)
print(seq)
print(len(seq) - 1, "steps")

## Answer
Why: 6 is even → 3; 3 is odd → 10; then 5, 16, 8, 4, 2, 1 — 8 steps.

In [ ]:
n = 6
seq = [n]
while n != 1:
    n = n // 2 if n % 2 == 0 else 3 * n + 1
    seq.append(n)
print(seq)
print(len(seq) - 1, "steps")

[6, 3, 10, 5, 16, 8, 4, 2, 1]
8 steps


## Exercise 2

(a) Why does `collatz(2**50)` finish so quickly although `2⁵⁰` is huge?
Can you tell **in advance** how many iterations it takes?

(b) The number of steps does not grow monotonically with `n`. Compare `n = 13` and `n = 27`.

(c) Write `check_collatz(k)` that runs the process (without printing) on every integer from 2 to `k`.
It halts if and only if the process halts on all of them. Try `k = 100_000` in class, `k = 1_000_000` at home.
Can you conclude that the process halts for **every** `n`?

In [ ]:
# Exercise 2(c): your solution here

def check_collatz(k):
    pass  # replace with your code

check_collatz(100_000)

## The 3n + 1 conjecture

**Collatz conjecture:** for every integer `n ≥ 1`, the process reaches 1 (the function halts).

- As of 2019 it was checked by computer for every `n` up to `87 · 2⁶⁰ ≈ 10²⁰`.
- That increases our confidence — but it **proves nothing**: the next number might be the exception.

## Exercise 3

(a) If you suspect that some integer `k` refutes the conjecture, why won't running `collatz(k)` help you **disprove** it?

(b) Why can't the following function help prove or disprove the conjecture?

```python
def check_all_collatz():
    n = 2
    while True:
        collatz(n)
        n = n + 1
```

(c) If some `m` never reaches 1, its sequence is either unbounded or bounded.
In the bounded case, how could a small change to `collatz` **detect** it? *Hint:* remember the values already seen.

## 10.3 The halting problem is undecidable

**Turing (1936):** nobody can write a program `Halt(f, x)` that, for **every** program `f` and input `x`,
correctly answers whether `f` halts on `x` (and always halts itself).

- Not because programmers are not clever enough, or languages are not powerful enough, or processors are too slow.
- It is a **fundamental** limitation of computing.
- Turing used an abstract model — the **Turing machine** — which can compute anything a modern computer can.
  So "not solvable by a Turing machine" means "not solvable by any program".

## Functions as inputs

The proof needs a function that receives **another function** as input.
In Python this is ordinary — such functions are called *higher-order functions*.

In [ ]:
# Code snippet: passing a function as an argument

def apply_twice(f, x):
    return f(f(x))

def add_three(n):
    return n + 3

print("apply_twice(add_three, 10) =", apply_twice(add_three, 10))
print("apply_twice(str.upper, 'acgt') =", apply_twice(str.upper, "acgt"))

## Functions as inputs
### What will be the output?

In [ ]:
def apply_twice(f, x):
    return f(f(x))

def double(n):
    return 2 * n

print(apply_twice(double, 5))
print(apply_twice(str.lower, "ACGT"))

## Answer
Why: a function can be passed like any other value: double(double(5)) = 20.

In [ ]:
def apply_twice(f, x):
    return f(f(x))

def double(n):
    return 2 * n

print(apply_twice(double, 5))
print(apply_twice(str.lower, "ACGT"))

20
acgt


## Proof by contradiction

Assume, for contradiction, that `Halt(f, x)` exists: it returns `True` if `f` halts on `x`, `False` otherwise,
and it always halts. (An error counts as halting.)

Using it, build `modify`:

```python
def modify(f):
    if Halt(f, f):          # f halts when given itself as input
        while True:         # ... then run forever!
            x = 0
    else:                   # f does not halt on itself
        return "yes, we can!"
```

Now ask: **does `modify(modify)` halt?**

## Both answers are impossible

1. **If `modify(modify)` does not halt**, it entered the infinite loop.
   That happens only if `Halt(modify, modify)` returned `True` — which says `modify(modify)` **does** halt. Contradiction.
2. **If `modify(modify)` halts**, it returned `"yes, we can!"`.
   That happens only if `Halt(modify, modify)` returned `False` — which says `modify(modify)` does **not** halt. Contradiction.

So `modify(modify)` halts **if and only if** it does not halt. The only assumption we made was that `Halt` exists —
so **`Halt` cannot exist**.

In [ ]:
# Code snippet: try it with ANY candidate Halt — the construction always
# defeats it

def make_modify(Halt):
    def modify(f):
        if Halt(f, f):
            return "loops forever"     # stands for: while True: pass
        else:
            return "yes, we can!"      # halts
    return modify

candidates = {
    "always says True":  lambda f, x: True,
    "always says False": lambda f, x: False,
}

for name, Halt in candidates.items():
    modify = make_modify(Halt)
    prediction = "halts" if Halt(modify, modify) else "loops forever"
    actual = "halts" if modify(modify) == "yes, we can!" else "loops forever"
    print(f"Halt that {name:18}: predicts modify(modify) {prediction:14} | "
          f"actually: {actual}")

## 10.4 Reflections on the proof

- It resembles the **liar's paradox**: "every statement I make is a lie".
- The idea of a program that refers to (a modified version of) itself is fundamental in mathematical logic:
  - **Cantor** (1845–1918): there are more real numbers than integers — the *diagonalization* technique.
  - **Gödel** (1906–1978): there are statements in number theory that can be neither proved nor disproved —
    *Gödel's incompleteness theorem*.

## And AI tools?

An AI coding assistant is itself a program running on a computer — so Turing's theorem applies to it too.

- It **cannot** decide, for every program, whether that program halts.
- It can often give a good answer for a specific program — but with no guarantee, and sometimes confidently wrong.
- When an AI tool tells you "this loop terminates", that is a claim to **check**, not a proof.

**AI changes who writes the code — not what is computable.**

## Chapter 10 Summary

In this chapter we studied:

- infinite loops, and why knowing whether a program halts matters
- the halting problem: given `f` and `x`, does `f` halt on `x`?
- the Collatz process — tested far and wide, still unproven
- Turing's theorem (1936): the halting problem is **undecidable** — no program can solve it
- the proof by contradiction with `modify(modify)`, and its relatives: the liar's paradox, Cantor, Gödel